# TechCare | Data Preprocessing - Phase 1

## Objective

Prepare the public customer-support ticket sample for loading into MySQL and subsequent analysis in Power BI.

The workflow includes:

- initial dataset inspection;
- validation of structure, missing values, and data types;
- column-name standardization;
- date conversion;
- temporal-consistency checks;
- text-field cleaning;
- export in a MySQL-compatible format.

**Input file:** `customer_support_tickets.csv`  
**Output file:** `customer_support_tickets_mysql.csv`

> Phase 1 is a limited public sample. It can reveal patterns and data-quality issues, but it cannot support definitive conclusions about automation.

## 1. Import Libraries and Load Data

Pandas is imported and the source file is loaded into a DataFrame. An initial review is then performed to understand its size, structure, and contents.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv ("customer_support_tickets.csv")

## 2. Initial Inspection

This section checks:

- the number of rows and columns;
- the original variable names;
- data types;
- a sample of records;
- initial descriptive statistics.

This review helps identify issues before any transformation is applied.

In [3]:
df.shape

(8469, 17)

In [4]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

In [6]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [7]:
df.describe(include="all")

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
count,8469.000000,8469,8469,8469.000000,8469,8469,8469,8469,8469,8469,8469,2769,8469,8469,5650,2769,2769.000000
unique,NaN,8028,8320,NaN,3,42,730,5,16,8077,3,2769,4,4,5470,2728,NaN
top,NaN,Michael Garcia,bsmith@example.com,NaN,Male,Canon EOS,2020-10-21,Refund request,Refund request,I'm having an issue with the {product_purchase...,Pending Customer Response,Case maybe show recently my computer follow.,Medium,Email,2023-06-01 15:21:42,2023-06-01 17:14:42,NaN
freq,NaN,5,4,NaN,2896,240,24,1752,576,25,2881,1,2192,2143,3,3,NaN
mean,4235.000000,NaN,NaN,44.026804,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.991333
std,2444.934048,NaN,NaN,15.296112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407016
min,1.000000,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000
25%,2118.000000,NaN,NaN,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000
50%,4235.000000,NaN,NaN,44.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000
75%,6352.000000,NaN,NaN,57.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000


### Initial Inspection Results

The dataset contains 8,469 tickets and 17 columns. Date variables were initially imported as text and must therefore be converted before durations and event sequences can be analysed.

Missing values are also present and must be assessed before deciding whether they are errors or are consistent with each ticket's status.

## 3. Data Quality and Missing Values

Missing values and the type of each variable are reviewed. The goal is not to remove nulls automatically, but to determine whether they represent information that should not yet exist, such as a resolution for a ticket that remains open.

In [8]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "python_dtype": df.dtypes,
    "non_null_values": df.notna().sum(),
    "null_values": df.isna().sum()
})

column_summary

,column,python_dtype,non_null_values,null_values
Ticket ID,Ticket ID,int64,8469,0
Customer Name,Customer Name,object,8469,0
Customer Email,Customer Email,object,8469,0
Customer Age,Customer Age,int64,8469,0
Customer Gender,Customer Gender,object,8469,0
Product Purchased,Product Purchased,object,8469,0
Date of Purchase,Date of Purchase,object,8469,0
Ticket Type,Ticket Type,object,8469,0
Ticket Subject,Ticket Subject,object,8469,0
Ticket Description,Ticket Description,object,8469,0


### 3.1 Identifier and Duplicate Validation

Each row should represent one ticket, and every ticket should have a populated, unique identifier. Fully duplicated rows, null identifiers, and repeated identifiers are checked explicitly.

In [9]:
identifier_validation = pd.Series({
    "duplicated_rows": df.duplicated().sum(),
    "null_ticket_ids": df["Ticket ID"].isna().sum(),
    "duplicated_ticket_ids": df["Ticket ID"].duplicated().sum()
})

identifier_validation

duplicated_rows        0
null_ticket_ids         0
duplicated_ticket_ids    0
dtype: int64

#### Result

No duplicated rows, null identifiers, or repeated identifiers are found. `Ticket ID` can therefore be used as the unique ticket identifier.

### 3.2 Missing Values by Ticket Status

The analysis checks whether missing first-response, resolution, and satisfaction values are consistent with ticket status. This distinguishes expected structural nulls from potential data-quality issues.

In [10]:
nulls_by_status = (
    df.groupby("Ticket Status")
    .agg(
        tickets=("Ticket ID", "size"),
        null_first_response=(
            "First Response Time",
            lambda column: column.isna().sum()
        ),
        null_resolution_time=(
            "Time to Resolution",
            lambda column: column.isna().sum()
        ),
        null_resolution=(
            "Resolution",
            lambda column: column.isna().sum()
        ),
        null_satisfaction=(
            "Customer Satisfaction Rating",
            lambda column: column.isna().sum()
        )
    )
)

nulls_by_status

,tickets,null_first_response,null_resolution_time,null_resolution,null_satisfaction
Ticket Status,,,,,
Closed,2769,0,0,0,0
Open,2819,2819,2819,2819,2819
Pending Customer Response,2881,0,2881,2881,2881


#### Result

Missing values are consistent with ticket status:

- closed tickets contain response, resolution, and satisfaction information;
- open tickets do not yet contain any of these values;
- pending tickets contain a first response but no resolution or satisfaction score.

These are structural nulls and should not be removed or imputed.

### 3.3 Numeric Variable Validation

Customer age and satisfaction ratings are checked against valid ranges.

- Age must be between 0 and 120 years.
- Satisfaction must be between 1 and 5 when populated.
- Null satisfaction values are retained because they correspond to tickets that have not yet been closed.

In [11]:
ages_out_of_range = (
    ~df["Customer Age"].between(0, 120)
).sum()

ratings_out_of_range = (
    df["Customer Satisfaction Rating"].notna()
    & ~df["Customer Satisfaction Rating"].between(1, 5)
).sum()

numeric_validation = pd.DataFrame({
    "variable": [
        "Customer Age",
        "Customer Satisfaction Rating"
    ],
    "minimum_value": [
        df["Customer Age"].min(),
        df["Customer Satisfaction Rating"].min()
    ],
    "maximum_value": [
        df["Customer Age"].max(),
        df["Customer Satisfaction Rating"].max()
    ],
    "null_values": [
        df["Customer Age"].isna().sum(),
        df["Customer Satisfaction Rating"].isna().sum()
    ],
    "out_of_range_values": [
        ages_out_of_range,
        ratings_out_of_range
    ]
})

numeric_validation

,variable,minimum_value,maximum_value,null_values,out_of_range_values
0,Customer Age,18.0,70.0,0,0
1,Customer Satisfaction Rating,1.0,5.0,5700,0


#### Result

Customer ages range from 18 to 70, and satisfaction ratings range from 1 to 5. No numeric values fall outside the defined ranges.

The 5,700 null satisfaction values correspond to tickets that have not yet been closed.

## 4. Column-Name Standardization

Original column names are converted to `snake_case`: lowercase words separated by underscores.

This simplifies MySQL queries and maintains a consistent naming convention across the project.

In [12]:
df_mysql = df.rename(columns={
    "Ticket ID": "ticket_id",
    "Customer Name": "customer_name",
    "Customer Email": "customer_email",
    "Customer Age": "customer_age",
    "Customer Gender": "customer_gender",
    "Product Purchased": "product_purchased",
    "Date of Purchase": "date_of_purchase",
    "Ticket Type": "ticket_type",
    "Ticket Subject": "ticket_subject",
    "Ticket Description": "ticket_description",
    "Ticket Status": "ticket_status",
    "Resolution": "resolution",
    "Ticket Priority": "ticket_priority",
    "Ticket Channel": "ticket_channel",
    "First Response Time": "first_response_time",
    "Time to Resolution": "time_to_resolution",
    "Customer Satisfaction Rating": "customer_satisfaction_rating"
})

df_mysql.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## 5. Date Conversion and Temporal Consistency

Date variables are converted from text to datetime values. This is required to order events, calculate durations, and verify that dates follow a logical sequence.

In [13]:
date_formats = {
    "date_of_purchase": "%Y-%m-%d",
    "first_response_time": "%Y-%m-%d %H:%M:%S",
    "time_to_resolution": "%Y-%m-%d %H:%M:%S"
}

nulls_before = df_mysql[list(date_formats)].isna().sum()

for column, date_format in date_formats.items():
    df_mysql[column] = pd.to_datetime(
        df_mysql[column],
        format=date_format,
        errors="coerce"
    )

nulls_after = df_mysql[list(date_formats)].isna().sum()

date_conversion_validation = pd.DataFrame({
    "nulls_before": nulls_before,
    "nulls_after": nulls_after,
    "new_nulls_from_conversion": nulls_after - nulls_before
})

date_conversion_validation

,nulls_before,nulls_after,new_nulls_from_conversion
date_of_purchase,0,0,0
first_response_time,2819,2819,0
time_to_resolution,5700,5700,0


### 5.1 Temporal-Consistency Validation

For closed tickets, the resolution timestamp should be equal to or later than the first response. The analysis counts records with a valid sequence and records with reversed timestamps.

Open and pending tickets are excluded because they do not yet have a resolution timestamp.

In [14]:
closed_tickets = df_mysql[
    df_mysql["ticket_status"] == "Closed"
].copy()

complete_dates = (
    closed_tickets["first_response_time"].notna()
    & closed_tickets["time_to_resolution"].notna()
)

closed_tickets_with_dates = closed_tickets[complete_dates]

temporal_validation = pd.Series({
    "closed_tickets": len(closed_tickets),
    "closed_with_complete_dates": len(closed_tickets_with_dates),
    "valid_temporal_sequence": (
        closed_tickets_with_dates["time_to_resolution"]
        >= closed_tickets_with_dates["first_response_time"]
    ).sum(),
    "reversed_timestamps": (
        closed_tickets_with_dates["time_to_resolution"]
        < closed_tickets_with_dates["first_response_time"]
    ).sum()
})

temporal_validation

closed_tickets                 2769
closed_with_complete_dates    2769
valid_temporal_sequence      1404
reversed_timestamps                1365
dtype: int64

#### Result

All 2,769 closed tickets contain a first response and a resolution timestamp. However, 1,365 have a resolution timestamp earlier than the first response.

These records are retained to preserve traceability to the source data, but they must be excluded from duration calculations.

In [15]:
df_mysql.shape

(8469, 17)

In [16]:
df_mysql.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaT,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaT,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## 6. MySQL Preparation

Line breaks are removed from text variables and maximum text lengths are reviewed. This supports appropriate SQL data types and prevents truncation during import.

In [17]:
text_columns = df_mysql.select_dtypes(include="object").columns

for column in text_columns:
    df_mysql[column] = (
        df_mysql[column]
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
    )

df_mysql.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaT,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaT,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [18]:
text_columns = df_mysql.select_dtypes(include="object").columns

text_lengths = pd.DataFrame({
    "column": text_columns,
    "maximum_length": [
        int(df_mysql[col].str.len().max())
        for col in text_columns
    ],
    "average_length": [
        df_mysql[col].str.len().mean().round(1)
        for col in text_columns
    ]
})

text_lengths

,column,maximum_length,average_length
0,customer_name,26,13.2
1,customer_email,34,21.8
2,customer_gender,6,5.0
3,product_purchased,30,14.3
4,ticket_type,20,15.8
5,ticket_subject,24,15.7
6,ticket_description,397,289.8
7,ticket_status,25,11.8
8,resolution,69,35.8
9,ticket_priority,8,5.3


## 7. Export and Final Validation

The transformed dataset is exported in a MySQL-compatible format. The file is then reloaded to confirm that the export preserves the number of rows, columns, identifiers, and null values.

In [19]:
df_mysql.to_csv(
    "customer_support_tickets_mysql.csv",
    index=False,
    na_rep="\\N"
)

In [20]:
df_export_check = pd.read_csv(
    "customer_support_tickets_mysql.csv",
    na_values=[r"\N"],
    keep_default_na=False
)

export_validation = pd.Series({
    "rows_before": len(df_mysql),
    "rows_after": len(df_export_check),
    "same_row_count": len(df_mysql) == len(df_export_check),
    "same_column_count": (
        len(df_mysql.columns) == len(df_export_check.columns)
    ),
    "same_column_names_and_order": (
        df_mysql.columns.tolist() == df_export_check.columns.tolist()
    ),
    "same_ticket_ids_and_order": (
        df_mysql["ticket_id"].astype(str).tolist()
        == df_export_check["ticket_id"].astype(str).tolist()
    ),
    "nulls_before": int(df_mysql.isna().sum().sum()),
    "nulls_after": int(df_export_check.isna().sum().sum())
})

export_validation

rows_before                            8469
rows_after                          8469
same_row_count                  True
same_column_count               True
same_column_names_and_order     True
same_ticket_ids_and_order               True
nulls_before                           19919
nulls_after                         19919
dtype: object

## Result

The export preserves all 8,469 rows, 17 columns, ticket identifiers, and null values from the transformed dataset.

The file is ready for loading into MySQL while retaining the original records and standardizing their structure.

Detected data-quality issues are not silently corrected. They are retained and documented so that downstream calculations and Power BI measures can handle them explicitly.